# Data analysis

Exploration of the raw sources (raw) before cleansing.
Findings go in [ANALYSIS.md](../ANALYSIS.md).

Sections:
1. Setup
2. Load raw sources
3. Generic profiling
4. Products analysis
5. Customers analysis
6. Orders analysis
7. Cross-source integrity




## 1. Setup

In [3]:
from pyspark.sql import SparkSession, functions as F


spark = (
    SparkSession.builder
    .appName("data_analysis")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/06 11:14:10 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.188.120.206 instead (on interface en0)
26/10/06 11:14:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 11:14:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:

RAW = "../data/raw"
SOURCES = {
    "products": f"{RAW}/products.csv",
    "customers": f"{RAW}/customers.csv",
    "orders": f"{RAW}/orders.csv",
}

## 2. Load raw sources

In [5]:
raw = {
    name: spark.read.option("header", True).csv(path)
    for name, path in SOURCES.items()
}

## 3. Generic profiling

In [6]:
def profile(df, name):
    """Row count, schema, nulls/blanks per column and duplicate rows."""
    total = df.count()
    print(f"==> {name}: {total:,} rows, {len(df.columns)} columns")
    df.printSchema()

    null_or_blank = [
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
    df.select(null_or_blank).show(truncate=False)

    print(f"exact duplicate rows: {total - df.distinct().count():,}")


for name, df in raw.items():
    profile(df, name)

==> products: 5,635 rows, 3 columns
root
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- UnitPrice: string (nullable = true)

+---------+-----------+---------+
|StockCode|Description|UnitPrice|
+---------+-----------+---------+
|0        |960        |0        |
+---------+-----------+---------+

exact duplicate rows: 0
==> customers: 4,389 rows, 2 columns
root
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)

+----------+-------+
|CustomerID|Country|
+----------+-------+
|9         |0      |
+----------+-------+

exact duplicate rows: 0
==> orders: 541,909 rows, 5 columns
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- CustomerID: string (nullable = true)

+---------+---------+--------+-----------+----------+
|InvoiceNo|StockCode|Quantity|InvoiceDate|CustomerID|
+---------+---------+------

## 4. Products analysis

Ingested daily as snapshots: the same `StockCode` is expected to appear on
several days. Check the snapshot date column and the price distribution.

### 4.1 Volume and key uniqueness

In [7]:
products = raw["products"]
print(
    "rows:", products.count(),
    "| distinct StockCode:",
    products.select("StockCode").distinct().count()
)

rows: 5635 | distinct StockCode: 3965


### 4.2 Price quality

In [8]:
products.select(
    F.min(F.col("UnitPrice").cast("double")).alias("min_price"),
    F.max(F.col("UnitPrice").cast("double")).alias("max_price"),
    F.sum(
        F.when(F.col("UnitPrice").cast("double") <= 0, 1).otherwise(0)
    ).alias("non_positive_prices"),
).show()

+---------+---------+-------------------+
|min_price|max_price|non_positive_prices|
+---------+---------+-------------------+
|      0.0|      5.0|                 10|
+---------+---------+-------------------+



### 4.3 Repeated StockCode investigation

In [9]:
products.groupBy("StockCode").agg(
    F.count("*").alias("rows"),
    F.countDistinct("Description").alias("descriptions"),
    F.countDistinct("UnitPrice").alias("prices")
).filter(
    F.col("rows") > 1
).orderBy(
    F.desc("rows")
).show(10)

+---------+----+------------+------+
|StockCode|rows|descriptions|prices|
+---------+----+------------+------+
|    23084|   8|           7|     8|
|    20713|   8|           7|     8|
|    21830|   7|           6|     7|
|    85175|   7|           6|     7|
|    21181|   6|           5|     6|
|    23131|   6|           5|     6|
|    22121|   5|           4|     5|
|    23343|   5|           5|     5|
|    22837|   5|           4|     5|
|   72807A|   5|           5|     5|
+---------+----+------------+------+
only showing top 10 rows


In [10]:
products.filter(
    F.col("StockCode") == "23084"
).show(truncate=False)

+---------+-----------------------------------+---------+
|StockCode|Description                        |UnitPrice|
+---------+-----------------------------------+---------+
|23084    |RABBIT NIGHT LIGHT                 |3.84     |
|23084    |NULL                               |0.6      |
|23084    |temp adjustment                    |4.46     |
|23084    |allocate stock for dotcom orders ta|2.79     |
|23084    |add stock to allocate online orders|0.83     |
|23084    |for online retail orders           |4.1      |
|23084    |Amazon                             |4.36     |
|23084    |website fixed                      |1.43     |
+---------+-----------------------------------+---------+



In [11]:
products.withColumn(
    "is_uppercase",
    F.col("Description") == F.upper(F.col("Description"))
).groupBy("is_uppercase").count().show()


products.filter(
    F.col("Description").isNotNull() &
    (F.col("Description") != F.upper(F.col("Description")))
).select(
    "StockCode", "Description", "UnitPrice"
).show(100, truncate=False)



+------------+-----+
|is_uppercase|count|
+------------+-----+
|        NULL|  960|
|        true| 4098|
|       false|  577|
+------------+-----+

+---------+-----------------------------------+---------+
|StockCode|Description                        |UnitPrice|
+---------+-----------------------------------+---------+
|10080    |check                              |0.12     |
|10133    |damaged                            |4.08     |
|16008    |check                              |0.05     |
|16045    |check                              |2.67     |
|18007    |ESSENTIAL BALM 3.5g TIN IN ENVELOPE|0.25     |
|20652    |check                              |0.37     |
|20665    |dotcom                             |0.7      |
|20665    |adjustment                         |3.58     |
|20711    |check                              |3.82     |
|20711    |lost in space                      |4.19     |
|20713    |wrongly marked. 23343 in box       |0.06     |
|20713    |wrongly coded-23343          

In [12]:
non_upper = products.filter(
    F.col("Description").isNotNull() &
    (F.col("Description") != F.upper(F.col("Description")))
)

print("Non-uppercase descriptions:", non_upper.count())

non_upper.groupBy("Description") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

Non-uppercase descriptions: 577
+----------------------+-----+
|Description           |count|
+----------------------+-----+
|check                 |146  |
|damages               |43   |
|damaged               |43   |
|found                 |25   |
|sold as set on dotcom |20   |
|adjustment            |15   |
|Damaged               |14   |
|Unsaleable, destroyed.|9    |
|thrown away           |9    |
|Amazon                |7    |
|Found                 |7    |
|dotcom                |6    |
|amazon                |5    |
|damages?              |5    |
|had been put aside    |5    |
|wet damaged           |5    |
|ebay                  |5    |
|smashed               |4    |
|missing               |3    |
|mailout               |3    |
+----------------------+-----+
only showing top 20 rows


In [13]:
product_case_flags = products.groupBy("StockCode").agg(
    F.max(
        F.when(
            F.col("Description") == F.upper(F.col("Description")),
            1
        ).otherwise(0)
    ).alias("has_upper"),

    F.max(
        F.when(
            F.col("Description") != F.upper(F.col("Description")),
            1
        ).otherwise(0)
    ).alias("has_non_upper")
)

product_case_flags.filter(
    (F.col("has_upper") == 1) &
    (F.col("has_non_upper") == 1)
).count()

417

In [14]:
product_case_flags.filter(
    (F.col("has_upper") == 0) &
    (F.col("has_non_upper") == 1)
).show(20, truncate=False)

+------------+---------+-------------+
|StockCode   |has_upper|has_non_upper|
+------------+---------+-------------+
|gift_0001_50|0        |1            |
|23595       |0        |1            |
|72038P      |0        |1            |
|gift_0001_20|0        |1            |
|84612B      |0        |1            |
|16162M      |0        |1            |
|gift_0001_40|0        |1            |
|46000R      |0        |1            |
|46000S      |0        |1            |
|20964       |0        |1            |
|72732       |0        |1            |
|21703       |0        |1            |
|gift_0001_10|0        |1            |
|79323G      |0        |1            |
|46000U      |0        |1            |
|23444       |0        |1            |
|B           |0        |1            |
|84611B      |0        |1            |
|DCGS0068    |0        |1            |
|M           |0        |1            |
+------------+---------+-------------+
only showing top 20 rows


In [15]:
non_upper_only_codes = (
    product_case_flags
    .filter(
        (F.col("has_upper") == 0) &
        (F.col("has_non_upper") == 1)
    )
    .select("StockCode")
)

products.join(
    non_upper_only_codes,
    "StockCode",
    "inner"
).orderBy("StockCode").show(20, truncate=False)

+---------+-----------------------------------+---------+
|StockCode|Description                        |UnitPrice|
+---------+-----------------------------------+---------+
|16151A   |FLOWERS HANDBAG blue and orange    |1.63     |
|16151A   |NULL                               |3.4      |
|16162M   |NULL                               |0.14     |
|16162M   |alan hodge cant mamage this section|1.14     |
|16162M   |THE KING GIFT BAG 25x24x12cm       |3.19     |
|18007    |ESSENTIAL BALM 3.5g TIN IN ENVELOPE|0.25     |
|18007    |NULL                               |0.94     |
|20954    |*USB Office Mirror Ball            |2.36     |
|20954    |NULL                               |4.34     |
|20964    |POLYESTER FILLER PAD 60x40cm       |2.9      |
|20964    |NULL                               |1.9      |
|21120    |*Boombox Ipod Classic              |4.47     |
|21120    |NULL                               |0.07     |
|21594    |Dr. Jam's Arouzer Stress Ball      |3.83     |
|21594    |NUL

## 5. Customers analysis

### 5.1 Volume and key uniqueness

In [16]:
customers = raw["customers"]
print("distinct CustomerID:", customers.select("CustomerID").distinct().count())

customers.groupBy("Country").count().orderBy(F.desc("count")).show(10, truncate=False)

distinct CustomerID: 4373
+--------------+-----+
|Country       |count|
+--------------+-----+
|United Kingdom|3951 |
|Germany       |95   |
|France        |88   |
|Spain         |31   |
|Belgium       |25   |
|Switzerland   |22   |
|Portugal      |20   |
|Italy         |15   |
|Finland       |12   |
|Austria       |11   |
+--------------+-----+
only showing top 10 rows


### 5.2 Conflicting customer-country mappings

In [17]:
customers.groupBy("CustomerID").agg(
    F.count("*").alias("rows"),
    F.countDistinct("Country").alias("countries")
).filter(
    F.col("rows") > 1
).orderBy(F.desc("rows")).show(20, truncate=False)

+----------+----+---------+
|CustomerID|rows|countries|
+----------+----+---------+
|NULL      |9   |9        |
|12394     |2   |2        |
|12429     |2   |2        |
|12457     |2   |2        |
|12431     |2   |2        |
|12417     |2   |2        |
|12370     |2   |2        |
|12455     |2   |2        |
|12422     |2   |2        |
+----------+----+---------+



In [18]:
customers.filter(
    F.col("CustomerID").isin(
        "12394", "12429", "12457", "12431",
        "12417", "12370", "12455", "12422"
    )
).orderBy("CustomerID").show(truncate=False)

+----------+-----------+
|CustomerID|Country    |
+----------+-----------+
|12370     |Cyprus     |
|12370     |Austria    |
|12394     |Belgium    |
|12394     |Denmark    |
|12417     |Belgium    |
|12417     |Spain      |
|12422     |Australia  |
|12422     |Switzerland|
|12429     |Austria    |
|12429     |Denmark    |
|12431     |Belgium    |
|12431     |Australia  |
|12455     |Spain      |
|12455     |Cyprus     |
|12457     |Switzerland|
|12457     |Cyprus     |
+----------+-----------+



## 6. Orders analysis

Things to look at: invoices starting with `C` (cancellations?), negative
quantities.

### 6.1 Volume and cardinality

In [19]:
orders = raw["orders"]
print("distinct InvoiceNo:", orders.select("InvoiceNo").distinct().count())

orders.select("InvoiceDate").show(5, truncate=False) 

distinct InvoiceNo: 25900
+---------------+
|InvoiceDate    |
+---------------+
|12/1/2010 8:45 |
|12/1/2010 9:45 |
|12/2/2010 14:23|
|12/3/2010 11:19|
|12/3/2010 11:28|
+---------------+
only showing top 5 rows


In [20]:
orders.select(
    F.countDistinct("InvoiceNo").alias("distinct_invoices"),
    F.countDistinct("StockCode").alias("distinct_stock_codes"),
    F.countDistinct("CustomerID").alias("distinct_customers")
).show()

+-----------------+--------------------+------------------+
|distinct_invoices|distinct_stock_codes|distinct_customers|
+-----------------+--------------------+------------------+
|            25900|                4070|              4372|
+-----------------+--------------------+------------------+



### 6.2 Quantity analysis

In [21]:
orders.select(
    F.min(F.col("Quantity").cast("int")).alias("min_quantity"),
    F.max(F.col("Quantity").cast("int")).alias("max_quantity"),
    F.sum(
        F.when(F.col("Quantity").cast("int") <= 0, 1).otherwise(0)
    ).alias("non_positive_quantity")
).show()

+------------+------------+---------------------+
|min_quantity|max_quantity|non_positive_quantity|
+------------+------------+---------------------+
|      -80995|       80995|                10624|
+------------+------------+---------------------+



### 6.3 Cancellation / negative quantity investigation

In [22]:
orders.select(
    F.sum(F.when(F.col("InvoiceNo").startswith("C"), 1).otherwise(0)).alias("cancellation_lines"),
    F.sum(F.when(F.col("Quantity").cast("int") < 0, 1).otherwise(0)).alias("negative_quantity_lines"),
).show()

+------------------+-----------------------+
|cancellation_lines|negative_quantity_lines|
+------------------+-----------------------+
|              9288|                  10624|
+------------------+-----------------------+



In [23]:
orders.filter(
    (F.col("Quantity").cast("int") <= 0) &
    (~F.col("InvoiceNo").startswith("C"))
).groupBy("StockCode").count() \
 .orderBy(F.desc("count")) \
 .show(30, truncate=False)

+---------+-----+
|StockCode|count|
+---------+-----+
|21830    |5    |
|85175    |5    |
|22719    |4    |
|82494L   |4    |
|72802C   |4    |
|85172    |4    |
|20713    |3    |
|23343    |3    |
|35954    |3    |
|85049D   |3    |
|23084    |3    |
|21135    |3    |
|40016    |3    |
|21161    |3    |
|46000S   |3    |
|22162    |3    |
|72807A   |3    |
|22034    |3    |
|84631    |3    |
|22837    |3    |
|20966    |3    |
|21621    |3    |
|20892    |3    |
|20774    |3    |
|22114    |3    |
|22423    |3    |
|22501    |3    |
|72807C   |3    |
|79163    |3    |
|72818    |3    |
+---------+-----+
only showing top 30 rows


In [24]:
orders.filter(
    (F.col("Quantity").cast("int") <= 0) &
    (~F.col("InvoiceNo").startswith("C"))
).select(
    "InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "CustomerID"
).show(30, truncate=False)

+---------+---------+--------+----------------+----------+
|InvoiceNo|StockCode|Quantity|InvoiceDate     |CustomerID|
+---------+---------+--------+----------------+----------+
|551429   |10002    |-3      |4/28/2011 15:05 |NULL      |
|576843   |10133    |-82     |11/16/2011 15:32|NULL      |
|549031   |10134    |-19     |4/5/2011 17:04  |NULL      |
|561086   |15036    |-530    |7/25/2011 9:32  |NULL      |
|576764   |16008    |-1510   |11/16/2011 13:10|NULL      |
|548615   |16010    |-7      |4/1/2011 11:47  |NULL      |
|551201   |16015    |-620    |4/27/2011 11:33 |NULL      |
|569379   |16015    |-69     |10/3/2011 16:33 |NULL      |
|556287   |16043    |-45     |6/10/2011 9:22  |NULL      |
|566768   |16045    |-3667   |9/14/2011 17:53 |NULL      |
|580382   |16045    |-900    |12/2/2011 17:59 |NULL      |
|560722   |16049    |-31     |7/20/2011 13:37 |NULL      |
|556286   |16052    |-16     |6/10/2011 9:22  |NULL      |
|554320   |16053    |-102    |5/23/2011 15:32 |NULL     

In [25]:
negative_non_cancelled = orders.filter(
    (F.col("Quantity").cast("int") <= 0) &
    (~F.col("InvoiceNo").startswith("C"))
)

negative_non_cancelled.groupBy(
    F.col("CustomerID").isNull().alias("customer_id_null")
).count().show()

+----------------+-----+
|customer_id_null|count|
+----------------+-----+
|            true| 1336|
+----------------+-----+



### 6.4 Candidate grain investigation

In [26]:
orders.groupBy("InvoiceNo", "StockCode").count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

+---------+---------+-----+
|InvoiceNo|StockCode|count|
+---------+---------+-----+
|555524   |22698    |20   |
|C544580  |S        |16   |
|555524   |22697    |12   |
|572861   |22775    |8    |
|C553531  |S        |7    |
|578289   |23395    |7    |
|C544583  |S        |7    |
|572344   |M        |7    |
|540524   |21756    |6    |
|541266   |21754    |6    |
|538514   |21756    |6    |
|541266   |21755    |6    |
|C558347  |S        |6    |
|574481   |23084    |6    |
|544463   |22256    |5    |
|576463   |22144    |5    |
|578289   |23393    |5    |
|545658   |16237    |5    |
|578289   |23394    |5    |
|555853   |22161    |5    |
+---------+---------+-----+
only showing top 20 rows


In [27]:
orders_dedup = orders.dropDuplicates()

duplicate_grain = (
    orders_dedup
    .groupBy("InvoiceNo", "StockCode")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Repeated (InvoiceNo, StockCode) after exact dedup:",
    duplicate_grain.count()
)

duplicate_grain.orderBy(F.desc("count")).show(20)

Repeated (InvoiceNo, StockCode) after exact dedup: 5086
+---------+---------+-----+
|InvoiceNo|StockCode|count|
+---------+---------+-----+
|   574965|    23084|    4|
|   571046|    22197|    4|
|   541266|    21621|    4|
|   568188|    84879|    4|
|   572703|    21818|    4|
|   580399|    21109|    4|
|   562549|    20829|    4|
|   581237|    22910|    3|
|   544177|    21363|    3|
|   573904|    23100|    3|
|   580399|    22367|    3|
|   580958|    23333|    3|
|   538514|    22086|    3|
|   578515|    22227|    3|
|   546035|    22801|    3|
|   578548|    84347|    3|
|   567184|    22696|    3|
|   578757|    22174|    3|
|   571039|    21034|    3|
|   572307|    21819|    3|
+---------+---------+-----+
only showing top 20 rows


In [28]:
orders.dropDuplicates().filter(
    (F.col("InvoiceNo") == "568188") &
    (F.col("StockCode") == "84879")
).show(truncate=False)


+---------+---------+--------+---------------+----------+
|InvoiceNo|StockCode|Quantity|InvoiceDate    |CustomerID|
+---------+---------+--------+---------------+----------+
|568188   |84879    |2       |9/25/2011 14:33|16049     |
|568188   |84879    |5       |9/25/2011 14:33|16049     |
|568188   |84879    |4       |9/25/2011 14:33|16049     |
|568188   |84879    |6       |9/25/2011 14:33|16049     |
+---------+---------+--------+---------------+----------+



## 7. Cross-source integrity

### 7.1 Customer integrity

In [29]:
orphan_customers = orders.join(customers, "CustomerID", "left_anti")
print("order lines with unknown CustomerID:", orphan_customers.count())

order lines with unknown CustomerID: 135080


In [30]:
# Unknown non-null customers
orders.filter(F.col("CustomerID").isNotNull()) \
    .join(
        customers.select("CustomerID").distinct(),
        "CustomerID",
        "left_anti"
    ).count()




0

### 7.2 Product integrity

In [31]:
orphan_products = orders.join(products, "StockCode", "left_anti")
print("order lines with unknown StockCode:", orphan_products.count())

orphan_products.select("StockCode").distinct().show(10, truncate=False)

order lines with unknown StockCode: 2041
+---------+
|StockCode|
+---------+
|85132b   |
|84884a   |
|47566b   |
|84661b   |
|84997d   |
|82001s   |
|84795b   |
|85049e   |
|85049a   |
|84536b   |
+---------+
only showing top 10 rows


In [32]:
# Which StockCodes are missing from products?
orders.select("StockCode").distinct() \
    .join(
        products.select("StockCode").distinct(),
        "StockCode",
        "left_anti"
    ).show(20, truncate=False)

+---------+
|StockCode|
+---------+
|85132b   |
|84884a   |
|47566b   |
|84661b   |
|84997d   |
|82001s   |
|84795b   |
|85049e   |
|85049a   |
|84536b   |
|84536a   |
|85184c   |
|46776b   |
|47518F   |
|72807b   |
|84031b   |
|84510c   |
|47593b   |
|84993a   |
|85049c   |
+---------+
only showing top 20 rows


### 7.3 StockCode normalization

In [33]:
orders_stockcodes = orders.select(
    F.upper(F.trim("StockCode")).alias("StockCode")
).distinct()

products_stockcodes = products.select(
    F.upper(F.trim("StockCode")).alias("StockCode")
).distinct()

unknown_stockcodes = orders_stockcodes.join(
    products_stockcodes,
    "StockCode",
    "left_anti"
)

print("Unknown StockCodes after normalization:", unknown_stockcodes.count())
unknown_stockcodes.show(100, truncate=False)

Unknown StockCodes after normalization: 0
+---------+
|StockCode|
+---------+
+---------+



## Analysis for data model

In [34]:
# Analyze how many distinct prices exist for each product
products.groupBy("StockCode").agg(
    F.countDistinct("UnitPrice").alias("distinct_prices")
).groupBy("distinct_prices").count().orderBy("distinct_prices").show()

+---------------+-----+
|distinct_prices|count|
+---------------+-----+
|              1| 2651|
|              2| 1061|
|              3|  185|
|              4|   48|
|              5|   14|
|              6|    2|
|              7|    2|
|              8|    2|
+---------------+-----+



In [36]:
# Measure the price range for each product to assess the magnitude of price variation
product_price_range = (
    products
    .withColumn("UnitPrice_num", F.col("UnitPrice").cast("double"))
    .groupBy("StockCode")
    .agg(
        F.countDistinct("UnitPrice_num").alias("distinct_prices"),
        F.min("UnitPrice_num").alias("min_price"),
        F.max("UnitPrice_num").alias("max_price")
    )
    .withColumn(
        "price_range",
        F.col("max_price") - F.col("min_price")
    )
)

product_price_range.filter(
    F.col("distinct_prices") > 1
).orderBy(F.desc("price_range")).show(30, truncate=False)

+---------+---------------+---------+---------+------------------+
|StockCode|distinct_prices|min_price|max_price|price_range       |
+---------+---------------+---------+---------+------------------+
|21192    |3              |0.05     |4.94     |4.890000000000001 |
|21870    |2              |0.06     |4.95     |4.890000000000001 |
|85198    |2              |0.09     |4.94     |4.8500000000000005|
|22618    |2              |0.0      |4.84     |4.84              |
|16207B   |2              |0.07     |4.83     |4.76              |
|22127    |3              |0.13     |4.88     |4.75              |
|85107    |3              |0.24     |4.96     |4.72              |
|21830    |7              |0.06     |4.74     |4.680000000000001 |
|79063D   |2              |0.16     |4.81     |4.6499999999999995|
|22130    |2              |0.35     |4.98     |4.630000000000001 |
|23135    |3              |0.04     |4.66     |4.62              |
|84865    |2              |0.13     |4.75     |4.62           

In [37]:
# Inspect all product records for StockCodes that have exactly two distinct prices
products_with_two_prices = (
    product_price_range
    .filter(F.col("distinct_prices") == 2)
    .select("StockCode")
)

products.join(
    products_with_two_prices,
    on="StockCode",
    how="inner"
).orderBy("StockCode", "UnitPrice").show(100, truncate=False)

+---------+-----------------------------------+---------+
|StockCode|Description                        |UnitPrice|
+---------+-----------------------------------+---------+
|10002    |NULL                               |2.73     |
|10002    |INFLATABLE POLITICAL GLOBE         |3.25     |
|10123C   |HEARTS WRAPPING TAPE               |0.37     |
|10123C   |NULL                               |4.1      |
|10133    |COLOURING PENCILS BROWN TUBE       |1.86     |
|10133    |damaged                            |4.08     |
|15030    |FAN BLACK FRAME                    |4.34     |
|15030    |NULL                               |4.48     |
|15036    |NULL                               |0.91     |
|15036    |ASSORTED COLOURS SILK FAN          |1.74     |
|15044B   |BLUE PAPER PARASOL                 |1.01     |
|15044B   |NULL                               |1.16     |
|15058B   |NULL                               |0.29     |
|15058B   |PINK POLKADOT GARDEN PARASOL       |4.28     |
|15058C   |ICE

In [38]:
# Check product price uniqueness after excluding records with a missing description
products_with_description = products.filter(
    F.col("Description").isNotNull()
)

products_with_description.groupBy("StockCode").agg(
    F.countDistinct("UnitPrice").alias("distinct_prices")
).groupBy("distinct_prices").count().orderBy("distinct_prices").show()

+---------------+-----+
|distinct_prices|count|
+---------------+-----+
|              1| 3207|
|              2|  512|
|              3|   97|
|              4|   25|
|              5|    5|
|              6|    2|
|              7|    2|
+---------------+-----+



In [39]:
# Inspect products that still have exactly two distinct prices after removing null descriptions
two_price_codes = (
    products_with_description
    .groupBy("StockCode")
    .agg(F.countDistinct("UnitPrice").alias("distinct_prices"))
    .filter(F.col("distinct_prices") == 2)
    .select("StockCode")
)

products_with_description.join(
    two_price_codes,
    on="StockCode",
    how="inner"
).orderBy("StockCode").show(100, truncate=False)

+---------+-----------------------------------+---------+
|StockCode|Description                        |UnitPrice|
+---------+-----------------------------------+---------+
|10080    |GROOVY CACTUS INFLATABLE           |4        |
|10080    |check                              |0.12     |
|10133    |COLOURING PENCILS BROWN TUBE       |1.86     |
|10133    |damaged                            |4.08     |
|15058A   |BLUE POLKADOT GARDEN PARASOL       |2.43     |
|15058A   |wet/rusty                          |3.74     |
|15058C   |ICE CREAM DESIGN GARDEN PARASOL    |1.37     |
|15058C   |wet/rusty                          |1.68     |
|16008    |SMALL FOLDING SCISSOR(POINTED EDGE)|2.76     |
|16008    |check                              |0.05     |
|16045    |POPART WOODEN PENCILS ASST         |3.56     |
|16045    |check                              |2.67     |
|16156L   |WRAP, CAROUSEL                     |1.07     |
|16156L   |WRAP CAROUSEL                      |1.47     |
|16162M   |ala

In [41]:
# Identify records whose full description exactly matches known operational terms
technical_terms = [
    "check",
    "damaged",
    "found",
    "adjustment",
    "amazon",
    "dotcom",
    "wet damaged",
    "wet/rusty",
    "stock check"
]

technical_records = products.filter(
    F.lower(F.trim(F.col("Description"))).isin(technical_terms)
)

technical_records.groupBy("Description").count().orderBy(
    F.desc("count")
).show(100, truncate=False)


# Measure remaining price ambiguity after removing null and known operational descriptions
clean_products_candidate = products.filter(
    F.col("Description").isNotNull() &
    ~F.lower(F.trim(F.col("Description"))).isin(technical_terms)
)

clean_products_candidate.groupBy("StockCode").agg(
    F.countDistinct("UnitPrice").alias("distinct_prices")
).groupBy("distinct_prices").count().orderBy(
    "distinct_prices"
).show()

+-----------+-----+
|Description|count|
+-----------+-----+
|check      |146  |
|damaged    |43   |
|found      |25   |
|adjustment |15   |
|Damaged    |14   |
|Amazon     |7    |
|Found      |7    |
|dotcom     |6    |
|amazon     |5    |
|wet damaged|5    |
|CHECK      |3    |
|AMAZON     |3    |
|Adjustment |2    |
|wet/rusty  |2    |
|stock check|2    |
|FOUND      |1    |
|Dotcom     |1    |
|DAMAGED    |1    |
+-----------+-----+

+---------------+-----+
|distinct_prices|count|
+---------------+-----+
|              1| 3412|
|              2|  366|
|              3|   51|
|              4|   14|
|              5|    4|
|              6|    2|
+---------------+-----+



In [42]:
# Measure price uniqueness after removing null and known operational descriptions
clean_products_candidate = products.filter(
    F.col("Description").isNotNull() &
    ~F.lower(F.trim(F.col("Description"))).isin(technical_terms)
)

price_distribution = (
    clean_products_candidate
    .groupBy("StockCode")
    .agg(F.countDistinct("UnitPrice").alias("distinct_prices"))
    .groupBy("distinct_prices")
    .count()
    .orderBy("distinct_prices")
)

price_distribution.show()

+---------------+-----+
|distinct_prices|count|
+---------------+-----+
|              1| 3412|
|              2|  366|
|              3|   51|
|              4|   14|
|              5|    4|
|              6|    2|
+---------------+-----+



In [43]:
# Measure the impact on order lines if products with multiple prices are excluded
ambiguous_products = (
    clean_products_candidate
    .groupBy("StockCode")
    .agg(F.countDistinct("UnitPrice").alias("distinct_prices"))
    .filter(F.col("distinct_prices") > 1)
    .select("StockCode")
)

orders_normalized = orders.withColumn(
    "StockCode_normalized",
    F.upper(F.trim(F.col("StockCode")))
)

ambiguous_products_normalized = ambiguous_products.withColumn(
    "StockCode_normalized",
    F.upper(F.trim(F.col("StockCode")))
)

affected_product_orders = (
    orders_normalized
    .join(
        ambiguous_products_normalized.select("StockCode_normalized"),
        "StockCode_normalized",
        "inner"
    )
)

print("Affected order lines:", affected_product_orders.count())
print("Total order lines:", orders.count())
print(
    "Percentage:",
    affected_product_orders.count() / orders.count() * 100
)

Affected order lines: 88961
Total order lines: 541909
Percentage: 16.41622486432224


In [44]:
# Measure the impact on order lines if customers associated with multiple countries are excluded
ambiguous_customers = (
    customers
    .filter(F.col("CustomerID").isNotNull())
    .groupBy("CustomerID")
    .agg(F.countDistinct("Country").alias("country_count"))
    .filter(F.col("country_count") > 1)
    .select("CustomerID")
)

affected_customer_orders = orders.join(
    ambiguous_customers,
    "CustomerID",
    "inner"
)

print("Affected order lines:", affected_customer_orders.count())
print("Total order lines:", orders.count())
print(
    "Percentage:",
    affected_customer_orders.count() / orders.count() * 100
)

Affected order lines: 926
Total order lines: 541909
Percentage: 0.17087739823475898


In [45]:
# Compare simple deterministic price-selection strategies for ambiguous products
price_strategies = (
    clean_products_candidate
    .groupBy("StockCode")
    .agg(
        F.min(F.col("UnitPrice").cast("double")).alias("min_price"),
        F.max(F.col("UnitPrice").cast("double")).alias("max_price"),
        F.avg(F.col("UnitPrice").cast("double")).alias("avg_price"),
        F.countDistinct("UnitPrice").alias("price_count")
    )
    .filter(F.col("price_count") > 1)
)

price_strategies.orderBy(F.desc("price_count")).show(30, truncate=False)

+---------+---------+---------+------------------+-----------+
|StockCode|min_price|max_price|avg_price         |price_count|
+---------+---------+---------+------------------+-----------+
|23084    |0.83     |4.46     |2.908333333333333 |6          |
|20713    |0.06     |4.14     |2.9916666666666667|6          |
|23343    |1.29     |4.84     |2.662             |5          |
|21830    |0.06     |4.74     |2.022             |5          |
|23131    |1.05     |4.85     |2.7339999999999995|5          |
|85175    |1.85     |4.85     |2.694             |5          |
|23236    |0.25     |2.31     |0.9875            |4          |
|79030G   |1.06     |2.47     |1.8625000000000003|4          |
|21621    |0.79     |4.17     |2.5275            |4          |
|72807A   |1.59     |4.52     |2.7699999999999996|4          |
|72802A   |0.73     |5.0      |2.435             |4          |
|23203    |0.27     |1.36     |0.8525            |4          |
|23196    |2.64     |3.96     |3.24              |4    

In [46]:
# Inspect ambiguous products to check whether the first source record looks like the canonical product record
products.filter(
    F.col("StockCode").isin(
        "23084",
        "20713",
        "23343",
        "21830",
        "23131",
        "85175"
    )
).show(100, truncate=False)

+---------+-----------------------------------+---------+
|StockCode|Description                        |UnitPrice|
+---------+-----------------------------------+---------+
|20713    |JUMBO BAG OWLS                     |4.14     |
|20713    |NULL                               |2.88     |
|20713    |wrongly marked. 23343 in box       |0.06     |
|20713    |wrongly coded-23343                |3.6      |
|20713    |found                              |1.31     |
|20713    |wrongly marked 23343               |4.1      |
|20713    |Marked as 23343                    |3.12     |
|20713    |wrongly coded 23343                |2.93     |
|21830    |ASSORTED CREEPY CRAWLIES           |0.06     |
|21830    |MERCHANT CHANDLER CREDIT ERROR, STO|0.09     |
|21830    |sold as 1                          |4.69     |
|21830    |?                                  |0.53     |
|21830    |damaged                            |4.2      |
|21830    |OOPS ! adjustment                  |4.74     |
|21830    |NUL